# Week 2: Kaplan-Meier and pairwise log-rank comparisons

This notebook reads bounded samples from `data/processed/cases_clean.parquet` with DuckDB, fits Kaplan-Meier curves, and runs pairwise log-rank tests. It does not load the 12-million-row cohort into pandas. The estimates below are sample estimates, not full-cohort results.

Nelson-Aalen, Cox PH, competing risks, and multiple-testing correction are intentionally outside this step.

In [ ]:
from pathlib import Path
import sys

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.survival_classical import fit_kaplan_meier, pairwise_logrank

PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cases_clean.parquet'
ROWS_PER_GROUP = 5_000
if not PROCESSED_PATH.exists():
    raise FileNotFoundError(PROCESSED_PATH)
print({'processed_path': str(PROCESSED_PATH), 'rows_per_group': ROWS_PER_GROUP})


## Bounded state sample

The window function limits the pandas result to 5,000 rows per state. DuckDB scans and filters the parquet; pandas receives only the bounded analysis frame.

In [ ]:
state_query = """
SELECT duration, event, state
FROM (
    SELECT duration, event, state,
           row_number() OVER (PARTITION BY state ORDER BY ddl_case_id) AS row_number
    FROM read_parquet(?)
)
WHERE row_number <= ?
ORDER BY state, row_number
"""

with duckdb.connect() as con:
    state_sample = con.execute(
        state_query, [str(PROCESSED_PATH), ROWS_PER_GROUP]
    ).fetchdf()

print({'rows': len(state_sample), 'groups': state_sample['state'].nunique()})
display(state_sample.groupby('state', sort=False).size().rename('rows').to_frame())


In [ ]:
state_km = fit_kaplan_meier(state_sample, group_col='state')
state_logrank = pairwise_logrank(state_sample, group_col='state')

state_summary = pd.DataFrame({
    'state': list(state_km),
    'median_days': [fitter.median_survival_time_ for fitter in state_km.values()],
})

print(f"Fitted {len(state_km)} state KM curves and {len(state_logrank)} pairwise tests")
display(state_summary)
display(state_logrank)


## Bounded case-type sample

The query selects the five largest non-null case-type groups before limiting each group. This avoids a large in-memory frame and avoids unstable comparisons among tiny categories.

In [ ]:
case_type_query = """
WITH top_case_types AS (
    SELECT case_type
    FROM read_parquet(?)
    WHERE case_type IS NOT NULL
    GROUP BY case_type
    ORDER BY count(*) DESC
    LIMIT 5
), ranked AS (
    SELECT p.duration, p.event, p.case_type,
           row_number() OVER (PARTITION BY p.case_type ORDER BY p.ddl_case_id) AS row_number
    FROM read_parquet(?) AS p
    INNER JOIN top_case_types AS t USING (case_type)
)
SELECT duration, event, case_type
FROM ranked
WHERE row_number <= ?
ORDER BY case_type, row_number
"""

with duckdb.connect() as con:
    case_type_sample = con.execute(
        case_type_query,
        [str(PROCESSED_PATH), str(PROCESSED_PATH), ROWS_PER_GROUP],
    ).fetchdf()

print({'rows': len(case_type_sample), 'groups': case_type_sample['case_type'].nunique()})
display(case_type_sample.groupby('case_type', sort=False).size().rename('rows').to_frame())


In [ ]:
case_type_km = fit_kaplan_meier(case_type_sample, group_col='case_type')
case_type_logrank = pairwise_logrank(case_type_sample, group_col='case_type')

case_type_summary = pd.DataFrame({
    'case_type': list(case_type_km),
    'median_days': [fitter.median_survival_time_ for fitter in case_type_km.values()],
})

print(f"Fitted {len(case_type_km)} case-type KM curves and {len(case_type_logrank)} pairwise tests")
display(case_type_summary)
display(case_type_logrank)


## Interpretation boundary

The tables are descriptive checks that the bounded Week 2 path works on the processed data. Because this notebook samples a fixed number of rows per group, do not treat its medians or p-values as weighted full-cohort estimates. A later analysis can define a prespecified full-cohort aggregation or sampling plan if those estimates are needed.